# Week 15 Optional: Advanced Agent Patterns

## Overview

This is a **supplementary notebook** for students who want to go deeper into agent
development. It covers advanced topics that go beyond the main session:

1. **Advanced Tool Patterns** — BaseTool subclass, tools with side effects, error handling
2. **DistilBERT as an Agent Tool** — Hybrid architecture combining a fine-tuned classifier with an LLM agent
3. **Agent Evaluation & Debugging** — Systematic evaluation across all transactions, step tracing
4. **Cost Optimization** — Triage pattern to reduce API costs

## Prerequisites

- Completed the **main Week 15 notebook** (required — we reuse `llm`, tools, and data from there)
- Familiarity with Week 14's fine-tuned DistilBERT model (for Section 2)

## Environment

**Platform**: AWS SageMaker (same as main notebook)

Run the setup cell below, then proceed to any section that interests you —
they are mostly independent.

In [ ]:
# =============================================================================
# SETUP — Same as main notebook (run this first!)
# =============================================================================
!pip install -q langchain langchain-aws langgraph transformers

import boto3, json, os
import sagemaker
from sagemaker import get_execution_role
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import tool, BaseTool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from typing import Type, Optional
from pydantic import BaseModel, Field

# SageMaker session — IAM role-based auth, no manual credentials
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

# Bedrock clients
bedrock_runtime = boto3.client('bedrock-runtime', region_name=AWS_REGION)
bedrock_client = boto3.client('bedrock', region_name=AWS_REGION)
BEDROCK_MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

# Initialize LLM
llm = ChatBedrockConverse(
    model=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.1,
    max_tokens=1024,
)

# ---- Reuse fraud data from main notebook ----
TRANSACTION_DATABASE = {
    "TXN-001": {"id": "TXN-001", "amount": 4500, "merchant": "Unknown Overseas Account", "type": "wire_transfer", "time": "03:47", "location": "International", "description": "Customer reports unauthorized wire transfer of $4,500 to unknown overseas account. No prior international transaction history. Transfer initiated at 3:47 AM local time.", "actual_label": "fraud"},
    "TXN-002": {"id": "TXN-002", "amount": 89.99, "merchant": "Netflix", "type": "subscription", "time": "10:00", "location": "Online", "description": "Regular monthly payment of $89.99 to Netflix streaming service. Consistent with 18-month subscription history.", "actual_label": "legitimate"},
    "TXN-003": {"id": "TXN-003", "amount": 1500, "merchant": "Multiple ATMs", "type": "atm_withdrawal", "time": "14:30", "location": "Multiple cities", "description": "Three consecutive ATM withdrawals totaling $1,500 in different cities within 2 hours. Card was reported lost the following day.", "actual_label": "fraud"},
    "TXN-004": {"id": "TXN-004", "amount": 234.56, "merchant": "Amazon.com", "type": "online_purchase", "time": "15:20", "location": "Online", "description": "Online purchase of $234.56 at Amazon.com for household electronics. Shipping to address on file.", "actual_label": "legitimate"},
    "TXN-005": {"id": "TXN-005", "amount": 2100, "merchant": "Luxury Jewelry Store", "type": "in_store", "time": "11:30", "location": "Miami", "description": "Customer disputes charge of $2,100 at luxury jewelry store in Miami. Customer's location confirmed as Chicago.", "actual_label": "fraud"},
    "TXN-006": {"id": "TXN-006", "amount": 3245.67, "merchant": "ABC Corp", "type": "direct_deposit", "time": "06:00", "location": "N/A", "description": "Automatic payroll direct deposit of $3,245.67 from employer ABC Corp. Matches bi-weekly pay schedule.", "actual_label": "legitimate"},
    "TXN-007": {"id": "TXN-007", "amount": 87.50, "merchant": "Various Digital Stores", "type": "online_purchase", "time": "22:15", "location": "Online", "description": "Multiple small online purchases ($5-$15) at various digital stores within 30 minutes. None in customer history.", "actual_label": "fraud"},
    "TXN-008": {"id": "TXN-008", "amount": 67.23, "merchant": "Whole Foods Market", "type": "in_store", "time": "17:45", "location": "Home area", "description": "Grocery purchase of $67.23 at Whole Foods Market. Customer shops here weekly.", "actual_label": "legitimate"},
    "TXN-009": {"id": "TXN-009", "amount": 8200, "merchant": "New Payee Transfer", "type": "wire_transfer", "time": "02:30", "location": "Foreign IP", "description": "Account password changed and $8,200 transferred to a new payee within 15 minutes. Login from foreign IP.", "actual_label": "fraud"},
    "TXN-010": {"id": "TXN-010", "amount": 3400, "merchant": "Electronics Store Lagos", "type": "in_store", "time": "16:00", "location": "Lagos, Nigeria", "description": "Credit card used for $3,400 purchase in Lagos, Nigeria. Cardholder has never traveled outside the US.", "actual_label": "fraud"},
}

CUSTOMER_HISTORY = {
    "TXN-001": {"avg_monthly_spend": 2500, "international_transactions": 0, "account_age_years": 5, "typical_hours": "8:00-22:00", "flagged_before": False},
    "TXN-002": {"avg_monthly_spend": 3200, "international_transactions": 0, "account_age_years": 3, "typical_hours": "7:00-23:00", "flagged_before": False},
    "TXN-003": {"avg_monthly_spend": 1800, "international_transactions": 2, "account_age_years": 7, "typical_hours": "9:00-21:00", "flagged_before": True},
    "TXN-004": {"avg_monthly_spend": 4100, "international_transactions": 5, "account_age_years": 10, "typical_hours": "6:00-00:00", "flagged_before": False},
    "TXN-005": {"avg_monthly_spend": 3500, "international_transactions": 1, "account_age_years": 4, "typical_hours": "8:00-22:00", "flagged_before": False},
    "TXN-006": {"avg_monthly_spend": 5000, "international_transactions": 3, "account_age_years": 8, "typical_hours": "6:00-23:00", "flagged_before": False},
    "TXN-007": {"avg_monthly_spend": 1200, "international_transactions": 0, "account_age_years": 2, "typical_hours": "9:00-21:00", "flagged_before": False},
    "TXN-008": {"avg_monthly_spend": 2800, "international_transactions": 1, "account_age_years": 6, "typical_hours": "7:00-22:00", "flagged_before": False},
    "TXN-009": {"avg_monthly_spend": 2200, "international_transactions": 0, "account_age_years": 4, "typical_hours": "8:00-20:00", "flagged_before": False},
    "TXN-010": {"avg_monthly_spend": 1500, "international_transactions": 0, "account_age_years": 3, "typical_hours": "9:00-21:00", "flagged_before": False},
}

FRAUD_POLICIES = {
    "international_first_time": "Flag and hold any first-time international transaction over $500.",
    "unusual_hours": "Transactions between 1:00 AM and 5:00 AM outside customer's typical pattern require enhanced monitoring.",
    "velocity_check": "More than 3 transactions within 30 minutes at different merchants triggers automatic review.",
    "geographic_mismatch": "Transaction location more than 500 miles from customer's last known location within 2 hours requires hold.",
    "amount_threshold": "Single transactions exceeding 3x the customer's average monthly spend require supervisor approval.",
    "new_payee_large_transfer": "Wire transfers over $5,000 to newly added payees require two-factor verification and 24-hour hold.",
    "card_testing_pattern": "Multiple small transactions ($0.01-$5.00) at different merchants within 10 minutes indicate card testing.",
}

# ---- Recreate tools from main notebook ----
@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID."""
    txn = TRANSACTION_DATABASE.get(transaction_id)
    if txn:
        return json.dumps({k: v for k, v in txn.items() if k != "actual_label"}, indent=2)
    return f"Transaction {transaction_id} not found."

@tool
def check_customer_history(transaction_id: str) -> str:
    """Check customer spending history and patterns for a given transaction."""
    history = CUSTOMER_HISTORY.get(transaction_id)
    if history:
        return json.dumps(history, indent=2)
    return f"No customer history found for {transaction_id}."

@tool
def calculate_risk_score(amount: float, avg_monthly_spend: float,
                         is_international: bool, is_unusual_hour: bool,
                         is_new_merchant: bool) -> str:
    """Calculate a fraud risk score (0-100) based on transaction characteristics."""
    amount_ratio = min(amount / max(avg_monthly_spend, 1), 5.0)
    amount_score = min(amount_ratio * 20, 100)
    intl_score = 90 if is_international else 0
    hour_score = 70 if is_unusual_hour else 0
    merchant_score = 50 if is_new_merchant else 0
    risk_score = amount_score * 0.30 + intl_score * 0.25 + hour_score * 0.20 + merchant_score * 0.15 + 10
    risk_level = "LOW" if risk_score < 30 else "MEDIUM" if risk_score < 60 else "HIGH"
    return json.dumps({"risk_score": round(risk_score, 1), "risk_level": risk_level}, indent=2)

@tool
def check_fraud_policy(policy_type: str) -> str:
    """Look up a specific fraud prevention policy by type."""
    if policy_type == "all":
        return json.dumps(FRAUD_POLICIES, indent=2)
    policy = FRAUD_POLICIES.get(policy_type)
    if policy:
        return json.dumps({"policy_type": policy_type, "rule": policy}, indent=2)
    return f"Policy '{policy_type}' not found. Available: {list(FRAUD_POLICIES.keys())}"

@tool
def get_similar_transactions(transaction_type: str) -> str:
    """Find all transactions of a given type to identify patterns."""
    similar = [
        {"id": txn["id"], "amount": txn["amount"]}
        for txn in TRANSACTION_DATABASE.values()
        if txn["type"] == transaction_type
    ]
    if similar:
        return json.dumps(similar, indent=2)
    return f"No transactions of type '{transaction_type}' found."

base_tools = [lookup_transaction, check_customer_history, calculate_risk_score,
              check_fraud_policy, get_similar_transactions]

print(f"✅ Setup complete! LLM: {BEDROCK_MODEL_ID}, Tools: {len(base_tools)}")
print(f"   Region: {AWS_REGION}, Transactions: {len(TRANSACTION_DATABASE)}")

# Section 1: Advanced Tool Patterns

## `@tool` vs `BaseTool`

In the main notebook, we used the `@tool` decorator — quick and easy. But sometimes
you need more control:

| Feature | `@tool` decorator | `BaseTool` subclass |
|---------|-------------------|---------------------|
| **Setup** | One-line decorator | Full class definition |
| **Validation** | Basic (type hints) | Custom with Pydantic |
| **Error handling** | Manual in function | Override `_run` method |
| **State** | Stateless | Can hold state (counters, caches) |
| **Best for** | Simple tools | Tools needing validation, state, or complex logic |

## Tools with Side Effects

So far, our tools only **read** data. But real agents also need to **write** —
flag a transaction, create a ticket, send an alert. Let's build tools that do both.

## Error Handling in Tools

What happens when a tool fails? The LLM sees the error message and can decide
what to do next — retry, try a different tool, or inform the user. Good error
messages help the agent recover gracefully.

In [ ]:
# =============================================================================
# DEMO: BaseTool Subclass — Tool with Validation and State
# =============================================================================
# This tool validates transaction IDs before querying, and tracks how many
# lookups have been performed (useful for cost monitoring).

class TransactionLookupInput(BaseModel):
    """Input schema for TransactionLookup tool — Pydantic validates automatically."""
    transaction_id: str = Field(description="Transaction ID in format TXN-XXX")

class TransactionLookupTool(BaseTool):
    """A more robust version of lookup_transaction with validation and state."""
    name: str = "validated_lookup_transaction"
    description: str = (
        "Look up transaction details by ID. Validates the ID format first. "
        "Transaction IDs must be in the format TXN-XXX (e.g., TXN-001)."
    )
    args_schema: Type[BaseModel] = TransactionLookupInput

    # State: track how many lookups we've done
    lookup_count: int = 0

    def _run(self, transaction_id: str) -> str:
        """Execute the tool with validation."""
        # Custom validation — this is the advantage of BaseTool
        if not transaction_id.startswith("TXN-"):
            return f"ERROR: Invalid transaction ID format '{transaction_id}'. Must start with 'TXN-'."

        self.lookup_count += 1

        txn = TRANSACTION_DATABASE.get(transaction_id)
        if txn:
            result = {k: v for k, v in txn.items() if k != "actual_label"}
            result["_meta"] = {"lookup_number": self.lookup_count}
            return json.dumps(result, indent=2)
        return f"Transaction {transaction_id} not found in database."

# Test it
validated_lookup = TransactionLookupTool()

# Valid ID
print("Valid lookup:")
print(validated_lookup._run("TXN-001")[:200])

# Invalid ID — see the helpful error
print("\nInvalid lookup:")
print(validated_lookup._run("INVALID-123"))

print(f"\nTotal lookups performed: {validated_lookup.lookup_count}")

In [ ]:
# =============================================================================
# DEMO: Tools with Side Effects — Flag and Escalate
# =============================================================================
# These tools simulate "writing" actions. In production, they'd update a
# database or send an API request. Here, we simulate with a simple list.

# Simulated "database" for flagged transactions and escalation tickets
flagged_transactions = []
escalation_tickets = []

@tool
def flag_transaction(transaction_id: str, reason: str) -> str:
    """Flag a transaction for manual review.

    Use this tool when your investigation reveals suspicious activity
    that requires human review. Provide the transaction ID and a clear
    reason for flagging.
    """
    flag_entry = {
        "transaction_id": transaction_id,
        "reason": reason,
        "status": "FLAGGED_FOR_REVIEW"
    }
    flagged_transactions.append(flag_entry)
    return json.dumps({
        "action": "FLAGGED",
        "transaction_id": transaction_id,
        "reason": reason,
        "message": f"Transaction {transaction_id} has been flagged for manual review."
    }, indent=2)


@tool
def escalate_to_supervisor(transaction_id: str, severity: str, summary: str) -> str:
    """Escalate a case to a supervisor for immediate attention.

    Use this for high-severity fraud cases that need immediate action.
    Severity should be 'HIGH', 'CRITICAL', or 'EMERGENCY'.
    """
    valid_severities = ["HIGH", "CRITICAL", "EMERGENCY"]
    if severity.upper() not in valid_severities:
        return f"ERROR: Severity must be one of {valid_severities}, got '{severity}'"

    ticket = {
        "ticket_id": f"ESC-{len(escalation_tickets) + 1:04d}",
        "transaction_id": transaction_id,
        "severity": severity.upper(),
        "summary": summary,
        "status": "OPEN"
    }
    escalation_tickets.append(ticket)
    return json.dumps({
        "action": "ESCALATED",
        "ticket": ticket,
        "message": f"Escalation ticket {ticket['ticket_id']} created for {transaction_id}."
    }, indent=2)


# Create an agent with side-effect tools
action_tools = base_tools + [flag_transaction, escalate_to_supervisor]

action_agent = create_react_agent(
    model=llm,
    tools=action_tools,
    prompt="You are a senior fraud analyst. After investigating a transaction, "
           "take appropriate action:\n"
           "- If FRAUD: flag the transaction AND escalate to supervisor\n"
           "- If NEEDS REVIEW: flag the transaction only\n"
           "- If LEGITIMATE: no action needed\n"
           "Always investigate thoroughly before taking action."
)

# Test: Investigate TXN-009 (fraud) — agent should flag AND escalate
result = action_agent.invoke(
    {"messages": [HumanMessage(content="Investigate TXN-009 and take appropriate action.")]}
)
print(result["messages"][-1].content[:500])

print(f"\n--- Side Effect Results ---")
print(f"Flagged transactions: {json.dumps(flagged_transactions, indent=2)}")
print(f"Escalation tickets: {json.dumps(escalation_tickets, indent=2)}")

# Section 2: DistilBERT as an Agent Tool — Hybrid Architecture

## Why Combine a Classifier with an Agent?

In Week 14, we fine-tuned a DistilBERT model that classifies transactions as
fraud or legitimate in **milliseconds** for **$0** (it runs locally). But it can
only classify — it can't explain *why* or check policies.

Our Bedrock agent can reason, use tools, and explain — but each investigation
costs multiple API calls (~$0.01-0.05 per investigation with Haiku).

**The hybrid approach**: Use DistilBERT as a *tool* that the agent can call.
The agent gets a fast, cheap first opinion, then uses its reasoning capabilities
to validate, investigate further, and explain the decision.

```
Hybrid Agent Flow:
  1. Agent receives investigation request
  2. Calls DistilBERT tool → gets quick classification + confidence
  3. If confidence > 0.9 → trusts the classifier, adds reasoning
  4. If confidence < 0.9 → investigates further with other tools
  5. Provides final verdict with full explanation
```

This pattern is common in production AI systems — cheap models handle the easy
cases, expensive models handle the hard ones.

In [ ]:
# =============================================================================
# DEMO: Wrap DistilBERT as an Agent Tool
# =============================================================================
# If you fine-tuned a model in Week 14, replace the model path below.
# We'll use a simulated classifier here so this notebook works standalone.

# Option A: If you have your Week 14 model saved:
# from transformers import pipeline
# classifier = pipeline("text-classification", model="./week14_fraud_model")

# Option B: Simulated classifier (works without Week 14 model)
# This simulates what a fine-tuned DistilBERT would return
import random
random.seed(42)

def simulated_classifier(text):
    """Simulate a fine-tuned fraud classifier based on keyword heuristics."""
    text_lower = text.lower()
    fraud_keywords = ["unauthorized", "unknown", "foreign", "stolen", "lost",
                      "multiple", "different cities", "overseas", "password changed"]
    legit_keywords = ["regular", "monthly", "consistent", "weekly", "payroll",
                      "on file", "history"]

    fraud_score = sum(1 for kw in fraud_keywords if kw in text_lower)
    legit_score = sum(1 for kw in legit_keywords if kw in text_lower)

    if fraud_score > legit_score:
        confidence = min(0.7 + fraud_score * 0.05, 0.98)
        return {"label": "FRAUD", "score": confidence}
    elif legit_score > fraud_score:
        confidence = min(0.7 + legit_score * 0.05, 0.98)
        return {"label": "LEGITIMATE", "score": confidence}
    else:
        return {"label": "UNCERTAIN", "score": 0.52}


@tool
def classify_with_distilbert(transaction_description: str) -> str:
    """Use a fine-tuned DistilBERT model to quickly classify a transaction.

    Returns the model's prediction (FRAUD/LEGITIMATE) and confidence score.
    This is a fast, free classification — use it as a starting point, then
    investigate further if the confidence is low (below 0.8).
    """
    result = simulated_classifier(transaction_description)
    return json.dumps({
        "model": "DistilBERT (fine-tuned)",
        "prediction": result["label"],
        "confidence": round(result["score"], 4),
        "note": "High confidence (>0.9) = trustworthy. Low confidence (<0.8) = investigate further."
    }, indent=2)

# Test on a few transaction descriptions
print("DistilBERT classifications:")
for txn_id in ["TXN-001", "TXN-002", "TXN-005"]:
    desc = TRANSACTION_DATABASE[txn_id]["description"]
    result = simulated_classifier(desc)
    actual = TRANSACTION_DATABASE[txn_id]["actual_label"]
    print(f"  {txn_id}: predicted={result['label']}, confidence={result['score']:.2f}, actual={actual}")

In [ ]:
# =============================================================================
# DEMO: Hybrid Agent — DistilBERT + LLM Reasoning
# =============================================================================
# The agent now has DistilBERT as an additional tool. It can get a quick
# classification, then decide whether to investigate further.

hybrid_tools = base_tools + [classify_with_distilbert]

hybrid_agent = create_react_agent(
    model=llm,
    tools=hybrid_tools,
    prompt="You are a senior fraud analyst with access to a fast ML classifier "
           "(classify_with_distilbert) AND investigation tools. Your workflow:\n"
           "1. First, get a quick classification from DistilBERT\n"
           "2. If confidence > 0.9, trust it but still verify with one more tool\n"
           "3. If confidence < 0.9, do a full investigation (lookup, history, risk, policies)\n"
           "4. Always provide your final VERDICT with reasoning that includes "
           "both the classifier's opinion and your investigation findings."
)

# Test on TXN-005 (fraud — geographic mismatch)
print("=" * 60)
print("HYBRID INVESTIGATION: TXN-005")
print("=" * 60)

result = hybrid_agent.invoke(
    {"messages": [HumanMessage(
        content="Investigate TXN-005. Start with the quick classifier, "
                "then investigate further if needed."
    )]}
)
print(result["messages"][-1].content[:600])

# Count how many tools were called
tool_calls = [m for m in result["messages"] if hasattr(m, 'tool_calls') and m.tool_calls]
print(f"\n💡 Agent made {sum(len(m.tool_calls) for m in tool_calls)} tool calls for this investigation")

# Section 3: Agent Evaluation & Debugging

## How to Evaluate Agents

Evaluating agents is harder than evaluating classifiers. We need to measure:

| Dimension | Question | How to Measure |
|-----------|----------|----------------|
| **Correctness** | Did it get the right answer? | Compare verdict to actual_label |
| **Efficiency** | How many tool calls? | Count tool call messages |
| **Completeness** | Did it check all relevant info? | Verify which tools were used |
| **Cost** | How much did it cost? | Count API calls x Haiku pricing |

## Step Tracing

When an agent gets something wrong, we need to trace through its reasoning:
- What tools did it call? In what order?
- What information did it see?
- Where did its reasoning go wrong?

This is like debugging code — but instead of stepping through lines, we step
through the agent's thought process.

In [ ]:
# =============================================================================
# DEMO: Evaluate Agent on All 10 Transactions
# =============================================================================
# Run the agent on every transaction, compare to ground truth, track metrics.

# Create a simple evaluation agent
eval_agent = create_react_agent(
    model=llm,
    tools=base_tools,
    prompt="You are a fraud analyst. Investigate the transaction and provide "
           "your verdict as EXACTLY one of: FRAUD, LEGITIMATE, or NEEDS_REVIEW. "
           "State your verdict clearly at the start of your response, like: "
           "'VERDICT: FRAUD' or 'VERDICT: LEGITIMATE'."
)

# Run evaluations
results = []
print("Running agent on all 10 transactions...")
print("=" * 70)

for txn_id in sorted(TRANSACTION_DATABASE.keys()):
    txn = TRANSACTION_DATABASE[txn_id]
    actual = txn["actual_label"]

    try:
        result = eval_agent.invoke(
            {"messages": [HumanMessage(content=f"Investigate {txn_id}. Is it fraud or legitimate?")]}
        )

        # Extract verdict from response
        response_text = result["messages"][-1].content.upper()
        if "VERDICT: FRAUD" in response_text or "VERDICT:FRAUD" in response_text:
            predicted = "fraud"
        elif "VERDICT: LEGITIMATE" in response_text or "VERDICT:LEGITIMATE" in response_text:
            predicted = "legitimate"
        elif "FRAUD" in response_text and "LEGITIMATE" not in response_text:
            predicted = "fraud"
        elif "LEGITIMATE" in response_text and "FRAUD" not in response_text:
            predicted = "legitimate"
        else:
            predicted = "uncertain"

        # Count tool calls and messages
        tool_call_count = sum(
            len(m.tool_calls) for m in result["messages"]
            if hasattr(m, 'tool_calls') and m.tool_calls
        )
        total_messages = len(result["messages"])

        correct = "✅" if predicted == actual else "❌"
        results.append({
            "txn_id": txn_id,
            "actual": actual,
            "predicted": predicted,
            "correct": predicted == actual,
            "tool_calls": tool_call_count,
            "messages": total_messages,
        })

        print(f"  {correct} {txn_id}: actual={actual:11s} predicted={predicted:11s} "
              f"tools={tool_call_count} msgs={total_messages}")

    except Exception as e:
        print(f"  ⚠️ {txn_id}: ERROR — {str(e)[:80]}")
        results.append({
            "txn_id": txn_id, "actual": actual, "predicted": "error",
            "correct": False, "tool_calls": 0, "messages": 0,
        })

# Summary statistics
correct_count = sum(1 for r in results if r["correct"])
total = len(results)
avg_tools = sum(r["tool_calls"] for r in results) / max(total, 1)
avg_msgs = sum(r["messages"] for r in results) / max(total, 1)

print(f"\n{'=' * 70}")
print(f"EVALUATION SUMMARY")
print(f"{'=' * 70}")
print(f"  Accuracy: {correct_count}/{total} ({100*correct_count/total:.0f}%)")
print(f"  Avg tool calls per investigation: {avg_tools:.1f}")
print(f"  Avg messages per investigation: {avg_msgs:.1f}")
print(f"  Estimated cost per investigation: ~${avg_tools * 0.005:.4f} (Haiku pricing)")

In [ ]:
# =============================================================================
# DEMO: Agent Debugging with Step Tracing
# =============================================================================
# Pick a transaction the agent might struggle with and trace every step.

def trace_agent_steps(agent, query: str):
    """Run an agent and print every step of its reasoning."""
    print(f"QUERY: {query}")
    print("=" * 60)

    step_num = 0
    for step in agent.stream(
        {"messages": [HumanMessage(content=query)]},
        stream_mode="updates",
    ):
        for node_name, node_output in step.items():
            step_num += 1
            messages = node_output.get("messages", [])

            for msg in messages:
                if node_name == "agent":
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        for tc in msg.tool_calls:
                            print(f"\n[Step {step_num}] 🔧 TOOL CALL: {tc['name']}")
                            print(f"  Args: {json.dumps(tc['args'])[:200]}")
                    elif hasattr(msg, 'content') and msg.content:
                        print(f"\n[Step {step_num}] 🧠 AGENT REASONING:")
                        print(f"  {msg.content[:400]}")

                elif node_name == "tools":
                    content = msg.content if isinstance(msg.content, str) else str(msg.content)
                    print(f"\n[Step {step_num}] 📊 TOOL RESULT ({msg.name}):")
                    print(f"  {content[:200]}")

    print(f"\n{'=' * 60}")
    print(f"Total steps: {step_num}")


# Trace TXN-007 — a subtle case (small purchases, card testing pattern)
trace_agent_steps(
    eval_agent,
    "Investigate TXN-007. The description mentions small purchases at various digital stores."
)

# Section 4: Cost Optimization for Agents

## The Triage Pattern

Not every transaction needs a full agent investigation. A smart production system
uses a **triage** approach:

```
Incoming Transaction
       |
       v
[DistilBERT Classifier]  ← Fast, free, runs locally
       |
  confidence > 0.9? ──YES──> Accept classifier verdict ($0)
       |
      NO
       v
[Full Agent Investigation]  ← Slow, costs API calls
       |
       v
  Final Verdict (~$0.01-0.05)
```

**Why this matters at scale:**
- 10,000 transactions/day
- If classifier handles 80% with high confidence: 8,000 free + 2,000 agent calls
- vs. running agent on all 10,000: 5x more expensive, 5x slower

## Cost Comparison

| Approach | Per Transaction | 10K/day | 1M/day |
|----------|----------------|---------|--------|
| **DistilBERT only** | ~$0.00 | ~$0 | ~$0 |
| **Agent only** | ~$0.03 | ~$300 | ~$30K |
| **Triage (80/20)** | ~$0.006 | ~$60 | ~$6K |

The triage pattern saves 80% of agent costs while maintaining accuracy on hard cases.

In [ ]:
# =============================================================================
# DEMO: Triage Pattern — Classifier First, Agent for Uncertain Cases
# =============================================================================

CONFIDENCE_THRESHOLD = 0.8  # Only invoke agent if classifier is uncertain

def triage_investigation(transaction_id: str):
    """Triage: use cheap classifier first, invoke agent only if needed."""
    txn = TRANSACTION_DATABASE[transaction_id]

    # Step 1: Quick classifier (free, instant)
    classifier_result = simulated_classifier(txn["description"])
    confidence = classifier_result["score"]
    prediction = classifier_result["label"]

    result = {
        "transaction_id": transaction_id,
        "classifier_prediction": prediction,
        "classifier_confidence": confidence,
        "used_agent": False,
        "agent_verdict": None,
        "final_verdict": None,
    }

    if confidence >= CONFIDENCE_THRESHOLD:
        # Classifier is confident — trust it
        result["final_verdict"] = prediction.lower()
        result["reasoning"] = f"Classifier confident ({confidence:.2f}) — no agent needed"
    else:
        # Classifier is uncertain — invoke the full agent
        result["used_agent"] = True
        agent_result = eval_agent.invoke(
            {"messages": [HumanMessage(content=f"Investigate {transaction_id}. Is it fraud?")]}
        )
        agent_text = agent_result["messages"][-1].content.upper()
        if "FRAUD" in agent_text and "LEGITIMATE" not in agent_text:
            result["agent_verdict"] = "fraud"
        elif "LEGITIMATE" in agent_text:
            result["agent_verdict"] = "legitimate"
        else:
            result["agent_verdict"] = "needs_review"
        result["final_verdict"] = result["agent_verdict"]
        result["reasoning"] = f"Classifier uncertain ({confidence:.2f}) — agent investigation"

    return result


# Run triage on all 10 transactions
print("TRIAGE EVALUATION — All 10 Transactions")
print("=" * 70)

triage_results = []
agent_invocations = 0

for txn_id in sorted(TRANSACTION_DATABASE.keys()):
    actual = TRANSACTION_DATABASE[txn_id]["actual_label"]
    result = triage_investigation(txn_id)
    triage_results.append(result)

    if result["used_agent"]:
        agent_invocations += 1

    correct = "✅" if result["final_verdict"] == actual else "❌"
    agent_flag = "🤖 AGENT" if result["used_agent"] else "⚡ FAST"
    print(f"  {correct} {txn_id}: actual={actual:11s} verdict={result['final_verdict']:11s} "
          f"{agent_flag} (conf={result['classifier_confidence']:.2f})")

# Summary
correct_count = sum(1 for r, txn_id in zip(triage_results, sorted(TRANSACTION_DATABASE.keys()))
                    if r["final_verdict"] == TRANSACTION_DATABASE[txn_id]["actual_label"])
total = len(triage_results)

print(f"\n{'=' * 70}")
print(f"TRIAGE SUMMARY")
print(f"{'=' * 70}")
print(f"  Accuracy: {correct_count}/{total} ({100*correct_count/total:.0f}%)")
print(f"  Agent invocations: {agent_invocations}/{total} ({100*agent_invocations/total:.0f}%)")
print(f"  Cost savings: {100*(1 - agent_invocations/total):.0f}% fewer API calls vs agent-on-everything")
print(f"  Estimated cost: {agent_invocations} agent calls x ~$0.03 = ~${agent_invocations * 0.03:.2f}")
print(f"  vs. all-agent: 10 x ~$0.03 = ~$0.30")

# Summary: Advanced Agent Patterns

## What We Covered

1. **Advanced Tools**: `BaseTool` subclass for validation, state tracking, and
   tools with side effects (flag, escalate)

2. **Hybrid Architecture**: DistilBERT classifier as an agent tool — the agent
   gets a fast first opinion before diving into a full investigation

3. **Agent Evaluation**: Systematic evaluation across all transactions measuring
   correctness, efficiency, and cost

4. **Cost Optimization**: Triage pattern — cheap classifier handles easy cases,
   expensive agent handles uncertain ones. Can save 60-80% of API costs.

## Key Insight

Production AI systems rarely use a single model or approach. The most effective
systems combine:
- **Fast, cheap models** for high-volume, straightforward decisions
- **Powerful agents** for complex cases that need reasoning and tool use
- **Human reviewers** for the highest-stakes decisions

This layered approach optimizes for cost, speed, and accuracy simultaneously.

## Next Week: LangGraph Deep Dive

In Week 16, we'll build on these agent fundamentals with LangGraph:
- Custom state machines for complex workflows
- Multi-agent systems with supervisor coordination
- Error handling, retries, and human-in-the-loop patterns